In [1]:
# generate embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/home/sanjay/Projects/Learning/RAG/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2991.14it/s]


In [2]:
import fitz
import os

def extractText(folder):
    documents = []

    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            path = os.path.join(folder, file)
            print(path)

            pdf = fitz.open(path)
            text = ""

            for page in pdf:
                text += page.get_text()
            
            documents.append({
                "source": path,
                "text": text
            })
    return documents

def chunkText(text, chunkSize=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunkSize
        chunks.append(text[start:end])
        start += chunkSize - overlap

    return chunks

# extract text from pdf
docs = extractText("Data")

# chunk text (overlapping chunks)
allChunks = []
for doc in docs:
    chunks = chunkText(doc['text'])
    for chunk in chunks:
        allChunks.append({
            "text": chunk,
            "source": doc['source']
        })

print(len(allChunks))
print(allChunks[0])

Data/ubuntu.pdf
1384
{'text': 'Praise for Previous Editions of \nThe Official Ubuntu Book\n“The Ofﬁcial Ubuntu Book is a great way to get you started with Ubuntu,\ngiving you enough information to be productive without overloading you.”\n—John Stevenson, DZone book reviewer\n“OUB is one of the best books I’ve seen for beginners.”\n—Bill Blinn, TechByter Worldwide\n“This book is the perfect companion for users new to Linux and Ubuntu.\nIt covers the basics in a concise and well-organized manner. General use\nis covered separately from ', 'source': 'Data/ubuntu.pdf'}


In [29]:

texts = [chunk['text'] for chunk in allChunks]
embeddings = model.encode(
    texts, 
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embeddings.shape)

(616, 384)


In [30]:
import faiss 
import pickle 

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

index.add(embeddings)

faiss.write_index(
    index, "faissIndex/index.bin"
)

with open("faissIndex/metadata.pkl", "wb") as f:
    pickle.dump(allChunks, f)


In [31]:
index = faiss.read_index("faissIndex/index.bin")

with open("faissIndex/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print(metadata[0])

{'text': 'Praise for Previous Editions of \nThe Official Ubuntu Book\n“The Ofﬁcial Ubuntu Book is a great way to get you started with Ubuntu,\ngiving you enough information to be productive without overloading you.”\n—John Stevenson, DZone book reviewer\n“OUB is one of the best books I’ve seen for beginners.”\n—Bill Blinn, TechByter Worldwide\n“This book is the perfect companion for users new to Linux and Ubuntu.\nIt covers the basics in a concise and well-organized manner. General use\nis covered separately from troubleshooting and error-handling, making\nthe book well-suited both for the beginner as well as the user that needs\nextended help.”\n—Thomas Petrucha, Austria Ubuntu User Group\n“I have recommended this book to several users who I instruct regularly on\nthe use of Ubuntu. All of them have been satisﬁed with their purchase and\nhave even been able to use it to help them in their journey along the way.”\n—Chris Crisafulli, Ubuntu LoCo Council, \nFlorida Local Community Team\n

In [32]:
query = "What is a kernel"

qemb = model.encode(
    [query], 
    convert_to_numpy=True,
    normalize_embeddings=True 
)
k = 3
score, index = index.search(qemb, k)

print(score)
print(index)

[[0.4445299 0.3346062 0.3297822]]
[[ 51 437  35]]


In [33]:
retrieved = []

for idx in index[0]:
    retrieved.append(
        metadata[idx]['text']
    )

for chunk in retrieved:
    print(chunk)
    print("="*50)


 recursive acronym that stands for “GNU’s Not UNIX.”
Linux
By the early 1990s, Stallman and a collection of other programmers work-
ing on GNU had developed a near-complete OS that could be freely shared.
They were, however, missing a ﬁnal essential piece in the form of a kernel—
a complex system command processor that lies at the center of any OS. In
1991, Linus Torvalds wrote an early version of just such a kernel, released it
under a free license, and called it Linux. Linus’s kernel was paired with the
GNU project’s development tools and OS and with the graphical window-
ing system called X. With this pairing, a completely free OS was born—free
both in terms of price and in Stallman’s terms of freedom.
All systems referred to as Linux today are, in fact, built on the work of this
collaboration. Technically, the term Linux refers only to the kernel. Many
programmers and contributors to GNU, including Stallman, argue
emphatically that the full OS should be referred to as GNU/Linux in 

In [34]:
context = "\n\n".join(retrieved)

prompt = f"""
You are a helpful assistant used in my RAG application.

Strictly answer ONLY using the provided context.

Context:
{context}
Query:
{query}

Answer: 
"""
print(prompt)


You are a helpful assistant used in my RAG application.

Strictly answer ONLY using the provided context.

Context:
 recursive acronym that stands for “GNU’s Not UNIX.”
Linux
By the early 1990s, Stallman and a collection of other programmers work-
ing on GNU had developed a near-complete OS that could be freely shared.
They were, however, missing a ﬁnal essential piece in the form of a kernel—
a complex system command processor that lies at the center of any OS. In
1991, Linus Torvalds wrote an early version of just such a kernel, released it
under a free license, and called it Linux. Linus’s kernel was paired with the
GNU project’s development tools and OS and with the graphical window-
ing system called X. With this pairing, a completely free OS was born—free
both in terms of price and in Stallman’s terms of freedom.
All systems referred to as Linux today are, in fact, built on the work of this
collaboration. Technically, the term Linux refers only to the kernel. Many
programmers an

In [35]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{
        "role": "user",
        "content": prompt
    }],
    temperature=0,
    max_tokens = 512

)

answer = response.choices[0].message.content

print(answer)

A kernel is a complex system command processor that lies at the center of any OS.
